In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, classification_report
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.multiclass import OneVsRestClassifier
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import VotingClassifier
import joblib

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
books = pd.read_parquet('//content//drive//MyDrive//AI//NLP//books2.pq')

In [5]:
# Всего 521 книга

books.shape

(521, 5)

In [6]:
books.title.nunique()

521

In [7]:
# 100 уникальных авторов

books.author.nunique()

100

In [8]:
def calculate_multiclass_metrics(y_true, y_pred):
    """
    Рассчитывает основные метрики для многоклассовой классификации.

    :param y_true: Список или массив истинных значений классов.
    :param y_pred: Список или массив предсказанных значений классов.
    :return: Словарь с основными метриками.
    """

    metrics = {
        'accuracy': round(accuracy_score(y_true, y_pred), 4),
        'precision_macro': round(precision_score(y_true, y_pred, average='macro', zero_division=np.nan), 4),
        'recall_macro': round(recall_score(y_true, y_pred, average='macro', zero_division=np.nan), 4),
        'f1_macro': round(f1_score(y_true, y_pred, average='macro', zero_division=np.nan), 4),
        'precision_micro': round(precision_score(y_true, y_pred, average='micro', zero_division=np.nan), 4),
        'recall_micro': round(recall_score(y_true, y_pred, average='micro', zero_division=np.nan), 4),
        'f1_micro': round(f1_score(y_true, y_pred, average='micro', zero_division=np.nan), 4),
    }

    # Также можно вывести подробный отчет по каждому классу
    report = classification_report(y_true, y_pred, zero_division=np.nan)
    print("Classification Report:\n", report)

    return metrics

In [9]:
class_column = 'author'

# Определяем количество экземпляров для каждого автора
class_counts = books[class_column].value_counts()

# Фильтруем только тех авторов, у которых более одного примера
filtered_class_counts = class_counts[class_counts > 1].index
filtered_books = books[books[class_column].isin(filtered_class_counts)]

# Стратифицированное разбиение
train_df, test_df = train_test_split(
    filtered_books,
    test_size=0.2,
    stratify=filtered_books[class_column],
    random_state=42
)

# Проверка распределения классов в train и test
print("Train class distribution:")
print(train_df[class_column].value_counts())

print("\nTest class distribution:")
print(test_df[class_column].value_counts())

Train class distribution:
author
Mark_Twain             33
Charles_Dickens        21
Alexandre_Dumas        20
Walter_Scott           20
H_G_Wells              18
                       ..
Jean_De_La_Fontaine     2
Bertolt_Brecht          2
Jean_Paul_Sartre        2
Mikhail_Lermontov       2
William_Faulkner        1
Name: count, Length: 70, dtype: int64

Test class distribution:
author
Mark_Twain                    8
Walter_Scott                  5
Charles_Dickens               5
H_G_Wells                     5
Alexandre_Dumas               5
Arthur_Conan_Doyle            5
William_Shakespeare           4
Robert_Sheckley               4
Jules_Verne                   3
Rudyard_Kipling               3
Mayne_Reid                    2
Philip_K_Dick                 2
Jack_London                   2
Fyodor_Dostoyevsky            2
J_K_Rowling                   2
Alexander_Pushkin             2
Robert_Louis_Stevenson        2
Oscar_Wilde                   2
Stephen_King                  2
Le

In [10]:
train_df.head(5)

,author,title,text,books_cnt,word_cnt
379,Nikolay_Gogol,The_Collected_Tales,THE COLLECTED TALES OF NIKOLAI GOGOL \n\nUKRAI...,3,157911
54,Aristophanes,The_Eleven_Comedies_Volume_2,"\r\n\r\n\r\n\r\nProduced by Jonathan Ingram, T...",7,104631
290,Jules_Verne,Round_The_Moon,﻿ FROM THE EARTH TO THE MOON...,14,90638
263,Johann_Wolfgang_Von_Goethe,Iphigenia_In_Tauris,"PERSONS OF THE DRAMA\n\nIPHIGENIA.\n\nTHOAS, K...",6,15981
351,Mark_Twain,The_Prince_And_The_Pauper,THE PRINCE AND THE PAUPER\n\nby Mark Twain\n\n...,41,69551


In [11]:
# Функция для обрезки первых 10 слов в строке
def remove_first_10_words(text):
    # Разделение строки на слова
    words = text.split()
    # Возврат строки без первых 10 слов
    return ' '.join(words[10:])

# Применение функции ко всем строкам столбца 'text'
X_train = train_df['text'].apply(remove_first_10_words)
X_test = test_df['text'].apply(remove_first_10_words)
y_train = train_df['author']
y_test = test_df['author']

Обучим лучшую модель логистичесой регрессии

In [ ]:
%%time

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
base_model = LogisticRegression(solver='liblinear',
                                random_state=33,
                                class_weight='balanced')
ovr = OneVsRestClassifier(base_model)
scaler = MaxAbsScaler()
pipeline = make_pipeline(vectorizer, scaler, ovr)

pipeline.fit(X_train, y_train)
lr_pred = pipeline.predict(X_test)

lr_pred_prob = pipeline.predict_proba(X_test)
lr_probs = lr_pred_prob[:, 1]

calculate_multiclass_metrics(y_test.tolist(), lr_pred)

Classification Report:
                             precision    recall  f1-score   support

           Agatha_Christie       1.00      1.00      1.00         1
             Aldous_Huxley       1.00      1.00      1.00         1
         Alexander_Pushkin       0.67      1.00      0.80         2
           Alexandre_Dumas       1.00      1.00      1.00         5
              Aristophanes       1.00      1.00      1.00         2
        Arthur_Conan_Doyle       1.00      1.00      1.00         5
                    Borges       1.00      1.00      1.00         1
           Charles_Dickens       1.00      1.00      1.00         5
           Dante_Alighieri       1.00      1.00      1.00         1
           Edgar_Allan_Poe       0.50      1.00      0.67         1
          Ernest_Hemingway       1.00      1.00      1.00         1
        F_Scott_Fitzgerald       1.00      1.00      1.00         1
        Fyodor_Dostoyevsky       1.00      1.00      1.00         2
    Gabriel_Garcia_Marq

{'accuracy': 0.9394,
 'precision_macro': 0.9577,
 'recall_macro': 0.905,
 'f1_macro': 0.8853,
 'precision_micro': 0.9394,
 'recall_micro': 0.9394,
 'f1_micro': 0.9394}

Обучим лучшую модель случайного леса

In [ ]:
%%time

# Векторизация признаков
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Масштабирование признаков
scaler = MaxAbsScaler()
X_train_scaled = scaler.fit_transform(X_train_tfidf)
X_test_scaled = scaler.transform(X_test_tfidf)

rf2 = RandomForestClassifier(
    bootstrap = True,
    random_state = 42,
    class_weight = "balanced",
    n_estimators = 1000
    )

rf2.fit(X_train_scaled, y_train)

y_pred2 = rf2.predict(X_test_scaled)
calculate_multiclass_metrics(y_test.tolist(), y_pred2)

Classification Report:
                             precision    recall  f1-score   support

           Agatha_Christie       1.00      1.00      1.00         1
             Aldous_Huxley       1.00      1.00      1.00         1
         Alexander_Pushkin        nan      0.00      0.00         2
           Alexandre_Dumas       0.71      1.00      0.83         5
              Aristophanes       1.00      1.00      1.00         2
        Arthur_Conan_Doyle       1.00      1.00      1.00         5
                    Borges       1.00      1.00      1.00         1
           Charles_Dickens       0.80      0.80      0.80         5
           Dante_Alighieri       1.00      1.00      1.00         1
           Edgar_Allan_Poe       1.00      1.00      1.00         1
          Ernest_Hemingway       1.00      1.00      1.00         1
        F_Scott_Fitzgerald       1.00      1.00      1.00         1
        Fyodor_Dostoyevsky       0.67      1.00      0.80         2
    Gabriel_Garcia_Marq

{'accuracy': 0.8081,
 'precision_macro': 0.9311,
 'recall_macro': 0.7193,
 'f1_macro': 0.7021,
 'precision_micro': 0.8081,
 'recall_micro': 0.8081,
 'f1_micro': 0.8081}

А теперь обучим ансамбль из этих моделей

In [12]:
%%time

# Определение базовых моделей
lr = LogisticRegression(solver='liblinear',
                        random_state=33,
                        class_weight='balanced')
rf = RandomForestClassifier(bootstrap = True,
                            random_state = 42,
                            class_weight = "balanced",
                            n_estimators = 1000)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
ovr_lr = OneVsRestClassifier(lr)
ovr_rf = OneVsRestClassifier(rf)
scaler = MaxAbsScaler()
lr_pipe = make_pipeline(vectorizer, scaler, ovr_lr)
rf_pipe = make_pipeline(vectorizer, scaler, ovr_rf)

# Создание ансамбля моделей
ensemble_model = VotingClassifier(
    estimators=[('logistic', lr_pipe), ('random_forest', rf_pipe)],
    voting='soft'
)

# Обучение ансамбля
ensemble_model.fit(X_train, y_train)

# Предсказание и оценка модели
y_pred_ens = ensemble_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_ens)

calculate_multiclass_metrics(y_test.tolist(), y_pred_ens)

Classification Report:
                             precision    recall  f1-score   support

           Agatha_Christie       1.00      1.00      1.00         1
             Aldous_Huxley       1.00      1.00      1.00         1
         Alexander_Pushkin       1.00      1.00      1.00         2
           Alexandre_Dumas       0.83      1.00      0.91         5
              Aristophanes       1.00      1.00      1.00         2
        Arthur_Conan_Doyle       1.00      1.00      1.00         5
                    Borges       1.00      1.00      1.00         1
           Charles_Dickens       1.00      1.00      1.00         5
           Dante_Alighieri       1.00      1.00      1.00         1
           Edgar_Allan_Poe       0.50      1.00      0.67         1
          Ernest_Hemingway       1.00      1.00      1.00         1
        F_Scott_Fitzgerald       1.00      1.00      1.00         1
        Fyodor_Dostoyevsky       1.00      1.00      1.00         2
    Gabriel_Garcia_Marq

{'accuracy': 0.9596,
 'precision_macro': 0.9722,
 'recall_macro': 0.92,
 'f1_macro': 0.9037,
 'precision_micro': 0.9596,
 'recall_micro': 0.9596,
 'f1_micro': 0.9596}

In [ ]:
%%time

# Работала 10 минут
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import make_pipeline

# Определение базовых моделей
lr = LogisticRegression(solver='liblinear',
                        random_state=33,
                        class_weight='balanced')
rf = RandomForestClassifier(bootstrap = True,
                            random_state = 42,
                            class_weight = "balanced",
                            n_estimators = 1000)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
ovr_lr = OneVsRestClassifier(lr)
ovr_rf = OneVsRestClassifier(rf)
scaler = MaxAbsScaler()
lr_pipe = make_pipeline(vectorizer, scaler, ovr_lr)
rf_pipe = make_pipeline(vectorizer, scaler, ovr_rf)

# Создание ансамбля моделей
ensemble_model = VotingClassifier(
    estimators=[('logistic', lr_pipe), ('random_forest', rf_pipe)],
    voting='soft'
)

# Обучение ансамбля
ensemble_model.fit(X_train, y_train)

CPU times: user 18min 27s, sys: 6.93 s, total: 18min 34s
Wall time: 17min 46s


VotingClassifier(estimators=[('logistic',
                              Pipeline(steps=[('tfidfvectorizer',
                                               TfidfVectorizer(max_features=10000,
                                                               ngram_range=(1,
                                                                            2))),
                                              ('maxabsscaler', MaxAbsScaler()),
                                              ('onevsrestclassifier',
                                               OneVsRestClassifier(estimator=LogisticRegression(class_weight='balanced',
                                                                                                random_state=33,
                                                                                                solver='liblinear')))])),
                             ('random_forest',
                              Pipeline(steps=[('tfidfvectorizer',
                                               TfidfVectorizer(max_features=10000,
                                                               ngram_range=(1,
                                                                            2))),
                                              ('maxabsscaler', MaxAbsScaler()),
                                              ('onevsrestclassifier',
                                               OneVsRestClassifier(estimator=RandomForestClassifier(class_weight='balanced',
                                                                                                    n_estimators=1000,
                                                                                                    random_state=42)))]))],
                 voting='soft')